In [2]:
import yfinance as yf
import pandas as pd
from scipy import stats

stocks = {
    "Nvidia" : "NVDA",
    "Apple" : "AAPL",
    "Microsoft" : "MSFT",
    "AMD" : "AMD",
    "Intel" : "INTC",
    "Cisco" : "CSCO",
    "Oracle" : "ORCL",
    "IBM" : "IBM",
    "Adobe" : "ADBE",
    "Salesforce" : "CRM",
    "Uber" : "UBER",
    "JP Morgan" : "JPM",
    "Visa" : "V",
    "Mastercard" : "MA",
    "Wells Fargo" : "WFC",
    "Morgan Stanley" : "MS",
    "American Express" : "AXP",
    "Bank of America" : "BAC",
    "Amazon" : "AMZN",
    "Tesla" : "TSLA",
    "Meta" : "META",
    "Google" : "GOOGL",
    "Netflix" : "NFLX",
    "Disney" : "DIS",
    "Verizon" : "VZ",
    "Costco" : "COST",
    "Sandisk" : "SNDK",
    "HP" : "HPE",
    "Motorola" : "MSI",
    "Airbnb" : "ABNB",
    "DoorDash" : "DASH"
}

actions = {
    "Overweight": "buy",
    "Outperform": "buy",
    "Buy": "buy",
    "Strong Buy": "buy",
    "Positive": "buy",
    "Market Outperform": "buy",
    "Sector Outperform": "buy",
    "Accumulate": "buy",
    "Outperformer": "buy",
    "Top Pick": "buy",
    "Long-Term Buy": "buy",
    "Neutral": "hold",
    "Hold": "hold",
    "Market Perform": "hold",
    "Equal-Weight": "hold",
    "Equal-weight": "hold",
    "Perform": "hold",
    "Peer Perform": "hold",
    "Sector Weight": "hold",
    "Sector Perform": "hold",
    "Fair Value": "hold",
    "Average": "hold",
    "In-Line": "hold",
    "Market Weight": "hold",
    "Mixed": "hold",
    "Underperform": "sell",
    "Underweight": "sell",
    "Reduce": "sell",
    "Sell": "sell",
    "Market Underperform": "sell",
    "Negative": "sell",
}

def get_forward_return(ticker, event_date, months):

    df = pricedata[ticker]
    df = df.copy()
    df.index = df.index.tz_localize(None)
    event_date = pd.to_datetime(event_date).tz_localize(None)
    target_date = event_date + pd.DateOffset(months = months)
    start_slice = df[df.index >= event_date]
    end_slice = df[df.index >= target_date]
    if start_slice.empty or end_slice.empty:
        return None
    start_price = start_slice.iloc[0]['Close']
    end_price = end_slice.iloc[0]['Close']

    return (end_price - start_price) / start_price

def run_tptest(df, rating_a, rating_b, horizon_col):

    a = df[df['rating'] == rating_a][horizon_col].dropna()
    b = df[df['rating'] == rating_b][horizon_col].dropna()
    t_stat, p_value = stats.ttest_ind(a, b, equal_var=False)
    tps.append({
        'comparison': f"{rating_a} vs {rating_b}",
        'horizon': horizon_col,
        't_stat': t_stat,
        'p_value': p_value
    })

def get_spy_return(event_date, months):
    df = spy_df.copy()
    df.index = df.index.tz_localize(None)
    event_date = pd.to_datetime(event_date).tz_localize(None)
    target_date = event_date + pd.DateOffset(months = months)
    start_slice = df[df.index >= event_date]
    end_slice = df[df.index <= event_date]
    if start_slice.empty or end_slice.empty:
        return None
    start_price = start_slice.iloc[0]['Close']
    end_price = end_slice.iloc[0]['Close']
    return (end_price - start_price) / start_price

pricedata = {}
for i in stocks.values():
    pricedata[i] = yf.Ticker(i).history(period="13y")

all = []
for i in stocks.values():
    df = yf.Ticker(i).upgrades_downgrades
    df = df.reset_index()
    df = df[df['ToGrade'] != '']
    df['rating'] = df['ToGrade'].map(actions)
    df = df.dropna(subset=['rating'])
    df['ticker'] = i
    all.append(df)

spy_df = yf.Ticker("^GSPC").history(period="13y")

tps = []

master = pd.concat(all, ignore_index=True)
master['ret_1m'] = master.apply(lambda row: get_forward_return(row['ticker'], row['GradeDate'], 1), axis=1)
master['ret_3m'] = master.apply(lambda row: get_forward_return(row['ticker'], row['GradeDate'], 3), axis=1)
master['ret_6m'] = master.apply(lambda row: get_forward_return(row['ticker'], row['GradeDate'], 6), axis=1)
master['spy_ret_1m'] = master['GradeDate'].apply(lambda d: get_spy_return(d, 1))
master['spy_ret_3m'] = master['GradeDate'].apply(lambda d: get_spy_return(d, 3))
master['spy_ret_6m'] = master['GradeDate'].apply(lambda d: get_spy_return(d, 6))
master['excess_ret_1m'] = master['ret_1m'] - master['spy_ret_1m']
master['excess_ret_3m'] = master['ret_3m'] - master['spy_ret_3m']
master['excess_ret_6m'] = master['ret_6m'] - master['spy_ret_6m']
run_tptest(master, 'buy', 'sell', 'excess_ret_1m')
run_tptest(master, 'buy', 'sell', 'excess_ret_3m')
run_tptest(master, 'buy', 'sell', 'excess_ret_6m')
run_tptest(master, 'buy', 'hold', 'excess_ret_1m')
run_tptest(master, 'buy', 'hold', 'excess_ret_3m')
run_tptest(master, 'buy', 'hold', 'excess_ret_6m')
run_tptest(master, 'hold', 'sell', 'excess_ret_1m')
run_tptest(master, 'hold', 'sell', 'excess_ret_3m')
run_tptest(master, 'hold', 'sell', 'excess_ret_6m')
tps_df = pd.DataFrame(tps)

/Users/pranavmandala/Development/projects/analyst-rating-accuracy/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [3]:
master

,GradeDate,Firm,ToGrade,FromGrade,Action,priceTargetAction,currentPriceTarget,priorPriceTarget,rating,ticker,ret_1m,ret_3m,ret_6m,spy_ret_1m,spy_ret_3m,spy_ret_6m,excess_ret_1m,excess_ret_3m,excess_ret_6m
0,2026-09-10 14:02:40,Piper Sandler,Overweight,,init,Announces,300.0,0.0,buy,NVDA,NaN,NaN,NaN,-0.778294,-0.778294,-0.778294,NaN,NaN,NaN
1,2026-09-04 11:38:09,Rosenblatt,Buy,Buy,main,Maintains,390.0,390.0,buy,NVDA,NaN,NaN,NaN,-0.778772,-0.778772,-0.778772,NaN,NaN,NaN
2,2026-09-04 11:04:59,Needham,Buy,Buy,reit,Maintains,300.0,300.0,buy,NVDA,NaN,NaN,NaN,-0.778772,-0.778772,-0.778772,NaN,NaN,NaN
3,2026-08-27 19:31:05,Citigroup,Buy,Buy,main,Raises,315.0,300.0,buy,NVDA,NaN,NaN,NaN,-0.779869,-0.779869,-0.779869,NaN,NaN,NaN
4,2026-08-27 18:33:14,Mizuho,Outperform,Outperform,main,Raises,315.0,300.0,buy,NVDA,NaN,NaN,NaN,-0.779869,-0.779869,-0.779869,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18384,2021-01-04 10:31:27,RBC Capital,Sector Perform,,init,Announces,135.0,0.0,hold,DASH,0.254795,-0.075331,0.265457,-0.544496,-0.544496,-0.544496,0.799291,0.469165,0.809953
18385,2021-01-04 10:08:32,Needham,Buy,,init,Announces,200.0,0.0,buy,DASH,0.254795,-0.075331,0.265457,-0.544496,-0.544496,-0.544496,0.799291,0.469165,0.809953
18386,2020-12-18 13:55:43,Susquehanna,Positive,,init,Announces,185.0,0.0,buy,DASH,0.240404,-0.157274,0.081009,-0.540558,-0.540558,-0.540558,0.780963,0.383284,0.621567
18387,2020-12-14 13:44:41,DA Davidson,Neutral,Buy,down,Raises,150.0,93.0,hold,DASH,0.177859,-0.114545,-0.001511,-0.540521,-0.540521,-0.540521,0.718380,0.425976,0.539011


In [4]:
tps_df

,comparison,horizon,t_stat,p_value
0,buy vs sell,excess_ret_1m,4.070874,5.165825e-05
1,buy vs sell,excess_ret_3m,1.797311,7.268261e-02
2,buy vs sell,excess_ret_6m,-1.658360,9.766006e-02
3,buy vs hold,excess_ret_1m,8.049397,9.673759e-16
4,buy vs hold,excess_ret_3m,5.573849,2.586067e-08
5,buy vs hold,excess_ret_6m,2.259309,2.389511e-02
6,hold vs sell,excess_ret_1m,0.634188,5.261135e-01
7,hold vs sell,excess_ret_3m,-0.626155,5.313661e-01
8,hold vs sell,excess_ret_6m,-2.527742,1.164695e-02
